# Create and run a local RAG pipeline from scratch

## Importing PDF from scratch

In [2]:
import os
import requests

# Get pdf path
pdf_path = "human_nutrition_text"

# Download pdf
if not os.path.exists(pdf_path):
    print(f"[INFO] File doesnt exist, downloading...")

    # Enter the URL of the PDF
    url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"
    
    # The local filename to save the downloaded file
    filename = pdf_path

    # Send a GET request to the URL
    response = requests.get(url)

    # Check if the request was successful
    if response.status_code == 200:
        # Open the file and save it
        with open(filename, "wb") as file:
            file.write(response.content)
        print(f"[INFO] The file has been downloaded and saved as {filename}")
    else:
        print(f"[INFO] Failed to download the file. Status code: {response.status_code}")
else:
    print(f"File {pdf_path} exists")



File human_nutrition_text exists


In [3]:
import fitz
from tqdm.auto import tqdm

def text_formatter(text:str) -> str:
    """ Performs minor formatting on text. """
    cleaned_text = text.replace("\n", " ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path : str) -> list[dict]:
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):
        text = page.get_text()
        text = text_formatter(text)
        pages_and_texts.append({"page_number":page_number - 41, 
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(". ")),
                                "page_token_count": len(text) / 4, 
                                "text": text})
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path = pdf_path)
pages_and_texts[:2]

                            
        

0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count_raw': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': -40,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [4]:
import random

random.sample(pages_and_texts, k=5)

[{'page_number': 422,
  'page_char_count': 1710,
  'page_word_count': 282,
  'page_sentence_count_raw': 16,
  'page_token_count': 427.5,
  'text': 'turnover rate. During exercise, especially when it is performed for  longer than two to three hours, muscle tissue is broken down and  some of the amino acids are catabolized to fuel muscle contraction.  To avert excessive borrowing of amino acids from muscle tissue to  synthesize energy during prolonged exercise, protein needs to be  obtained from the diet. Intense exercise, such as strength training,  stresses muscle tissue so that afterward, the body adapts by  building bigger, stronger, and healthier muscle tissue. The body  requires protein post-exercise to accomplish this. The IOM does  not set different RDAs for protein intakes for athletes, but the AND,  the American College of Sports Medicine, and Dietitians of Canada  have the following position statements4:  Nitrogen balance studies suggest that dietary protein intake  necessary 

In [6]:
import pandas as pd
df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,147,3,199.25,Contents Preface University of Hawai‘i at Mā...


In [7]:
df.describe()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,1208.00000,1208.000000,1208.000000,1208.000000,1208.000000
mean,562.50000,1148.004139,199.499172,10.519868,287.001035
std,348.86387,560.382275,95.830681,6.548495,140.095569
min,-41.00000,0.000000,1.000000,1.000000,0.000000
25%,260.75000,762.000000,134.000000,5.000000,190.500000
50%,562.50000,1231.500000,216.000000,10.000000,307.875000
75%,864.25000,1603.500000,272.000000,15.000000,400.875000
max,1166.00000,2308.000000,430.000000,39.000000,577.000000


### Further text processing (splitting pages into sentences)

Two ways to do this : 
1. Do this by splitting on `". "`.
2. We can do this with a NLP library like Spacy or NLTK

In [23]:
from spacy.lang.en import English

nlp = English()

# Add a sentencizer pipeline
nlp.add_pipe("sentencizer")

# Create document instance as an example
doc = nlp("This is a sentence. This is another sentence. I like potatoes.")
assert len(list(doc.sents)) == 3

# Print out sentences split
list(doc.sents)


[This is a sentence., This is another sentence., I like potatoes.]

In [26]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)

    # make sure all sentences are strings. The default type is spacy datatype
    item["sentence"] = [str(sentence) for sentence in item["sentences"]]

    # count the sentences
    item["page_sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [27]:
random.sample(pages_and_texts, k=1)

[{'page_number': 1045,
  'page_char_count': 308,
  'page_word_count': 53,
  'page_sentence_count_raw': 3,
  'page_token_count': 77.0,
  'text': 'effectively. By contrast, eating a variety of foods from all food groups  fuels the body by providing what it needs to produce energy,  promote metabolic activity, prevent micronutrient deficiencies,  ward off chronic disease, and  bolstering a sense of overall health  and well-being.  Introduction  |  1045',
  'sentences': [effectively.,
   By contrast, eating a variety of foods from all food groups  fuels the body by providing what it needs to produce energy,  promote metabolic activity, prevent micronutrient deficiencies,  ward off chronic disease, and  bolstering a sense of overall health  and well-being.,
    Introduction  |  1045],
  'sentence': ['effectively.',
   'By contrast, eating a variety of foods from all food groups  fuels the body by providing what it needs to produce energy,  promote metabolic activity, prevent micronutrient d

In [28]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00,10.32
std,348.86,560.38,95.83,6.55,140.10,6.30
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,5.00,190.50,5.00
50%,562.50,1231.50,216.00,10.00,307.88,10.00
75%,864.25,1603.50,272.00,15.00,400.88,15.00
max,1166.00,2308.00,430.00,39.00,577.00,28.00


### Chunking our sentences together 

The concept of splitting larger pieces of text into smaller ones is often referred to as text splitting or chunking.
    We'll split into groups of 10 sentences.

There are frameworks (LangChain) to do this but we will be using pure python for this. 

In [30]:
# Define split size to turn group of sentences to chunks
num_sentences_chunk_size = 10

# Create a function to split lists of texts recursively into chunk size
# e.g. [20] -> [10, 10] or [25] -> [10], [10], [5]

def split_list(input_list: list[str],
               slice_size: int=num_sentences_chunk_size) -> list[list[str]]:
    return [input_list[i:i+slice_size] for i in range(0, len(input_list), slice_size)]

test_list = list(range(25))
split_list(test_list)

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
 [20, 21, 22, 23, 24]]

In [31]:
# Loop through pages and texts and split sentences into chunks
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list = item["sentences"],
                                         slice_size = num_sentences_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [38]:
random.sample(pages_and_texts, k = 1)

[{'page_number': 938,
  'page_char_count': 1151,
  'page_word_count': 199,
  'page_sentence_count_raw': 9,
  'page_token_count': 287.75,
  'text': 'Image by  Cosmed /  CC BY-SA  3.0  Muscle Strength  Muscle strength is developed and maintained by weight or  resistance training that often is called anaerobic exercise. Anaerobic  exercise consists of short duration, high intensity movements that  rely on immediately available energy sources and require little or  no oxygen during the activity. This type of high intensity training  is used to build muscle strength by short, high intensity activities.  Building muscle mass is not just crucial for athletes and  bodybuilders—building muscle strength and endurance is important  for children, seniors, and everyone in between. The support that  your muscles provide allows you to work, play, and live more  efficiently. Strength training involves the use of resistance  machines, resistance bands, free weights, or other tools. However,  you do not

In [43]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(1)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,1208.0,1208.0,1208.0,1208.0,1208.0,1208.0,1208.0
mean,562.5,1148.0,199.5,10.5,287.0,10.3,1.5
std,348.9,560.4,95.8,6.5,140.1,6.3,0.6
min,-41.0,0.0,1.0,1.0,0.0,0.0,0.0
25%,260.8,762.0,134.0,5.0,190.5,5.0,1.0
50%,562.5,1231.5,216.0,10.0,307.9,10.0,1.0
75%,864.2,1603.5,272.0,15.0,400.9,15.0,2.0
max,1166.0,2308.0,430.0,39.0,577.0,28.0,3.0


### Splitting each chunk into its own item

We'd like to embed each chunk of sentences into its own numerical representation.

In [75]:
import re

# Split each chunk into its own item
pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]

        # Join the sentences together into a paragraph-like structure, i.e., join the list of sentences into one paragraph
        joined_sentence_chunk = "".join([span.text for span in sentence_chunk]).strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk)
        
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        # Getting a stat on our chunks
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4   # 1 token = ~4 chars

        pages_and_chunks.append(chunk_dict)
len(pages_and_chunks)

  0%|          | 0/1208 [00:00<?, ?it/s]

1843

In [77]:
random.sample(pages_and_chunks, k=1)

[{'page_number': 31,
  'sentence_chunk': 'Adequacy  An adequate diet is one that favors nutrient-dense foods. Nutrient- dense foods are defined as foods that contain many essential  nutrients per calorie. Nutrient-dense foods are the opposite of  “empty-calorie” foods, such as sugary carbonated beverages, which  are also called “nutrient-poor.”Nutrient-dense foods include fruits  and vegetables, lean meats, poultry, fish, low-fat dairy products, and  whole grains. Choosing more nutrient-dense foods will facilitate  weight loss, while simultaneously providing all necessary nutrients. Balance  Balance the foods in your diet. Achieving balance in your diet entails  not consuming one nutrient at the expense of another. For example,  calcium is essential for healthy teeth and bones, but too much  calcium will interfere with iron absorption. Most foods that are  good sources of iron are poor sources of calcium, so in order to  get the necessary amounts of calcium and iron from your diet, a  

In [78]:
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,1843.00,1843.00,1843.00,1843.00
mean,583.38,749.91,128.55,187.48
std,347.79,455.68,80.01,113.92
min,-41.00,14.00,4.00,3.50
25%,280.50,321.50,53.00,80.38
50%,586.00,762.00,132.00,190.50
75%,890.00,1137.50,195.00,284.38
max,1166.00,1870.00,415.00,467.50


In [79]:
df.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count
0,-41,Human Nutrition: 2020 Edition,29,4,7.25
1,-39,Human Nutrition: 2020 Edition UNIVERSITY OF ...,320,54,80.00
2,-38,Human Nutrition: 2020 Edition by University of...,212,32,53.00
3,-37,Contents Preface University of Hawai‘i at Mā...,797,147,199.25
4,-36,Lifestyles and Nutrition University of Hawai‘...,976,179,244.00


### Filter chunks of text for short chunks

Chunks that do not contain much useful information

In [81]:
min_token_length = 30
for row in df[df["chunk_token_count"] <= min_token_length].sample(5).iterrows():
    print(f'Chunk token count: {row[1]["chunk_token_count"]} | Text : {row[1]["sentence_chunk"]}')
    

Chunk token count: 13.75 | Text : https://doi.org/10.1186/ 1743-7075-4-24. Sulfur  |  637
Chunk token count: 29.5 | Text : 2010). EH. Net Encyclopedia. http://eh.net/?s=History+of+Food+and+Drug+Regulatio Protecting the Public Health  |  1011
Chunk token count: 14.25 | Text : PART IX  CHAPTER 9. VITAMINS  Chapter 9. Vitamins  |  513
Chunk token count: 3.75 | Text : 622  |  Calcium
Chunk token count: 24.5 | Text : view it online here:  http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=153    194  |  Chloride


In [83]:
pages_and_chunks_over_min_token_length = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_token_length[:2]

[{'page_number': -39,
  'sentence_chunk': 'Human Nutrition: 2020  Edition  UNIVERSITY OF HAWAI‘I AT MĀNOA  FOOD SCIENCE AND HUMAN  NUTRITION PROGRAM  ALAN TITCHENAL, SKYLAR HARA,  NOEMI ARCEO CAACBAY, WILLIAM  MEINKE-LAU, YA-YUN YANG, MARIE  KAINOA FIALKOWSKI REVILLA,  JENNIFER DRAPER, GEMADY  LANGFELDER, CHERYL GIBBY, CHYNA  NICOLE CHUN, AND ALLISON  CALABRESE',
  'chunk_char_count': 320,
  'chunk_word_count': 54,
  'chunk_token_count': 80.0},
 {'page_number': -38,
  'sentence_chunk': 'Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and  Human Nutrition Program is licensed under a Creative Commons Attribution 4.0  International License, except where otherwise noted.',
  'chunk_char_count': 212,
  'chunk_word_count': 32,
  'chunk_token_count': 53.0}]

In [92]:
random.sample(pages_and_chunks_over_min_token_length, k=1)

[{'page_number': 157,
  'sentence_chunk': 'Water is the foundation of all life—the surface of the earth is 70  percent water; the volume of water in humans is about 60 percent. Water As a Transportation Vehicle  Water is called the “universal solvent” because more substances  dissolve in it than any other fluid. Molecules dissolve in water  because of the hydrogen and oxygen molecules ability to loosely  bond with other molecules. Molecules of water (H2O) surround  substances, suspending them in a sea of water molecules. The  solvent action of water allows for substances to be more readily  transported. A pile of undissolved salt would be difficult to move  throughout tissues, as would a bubble of gas or a glob of fat. Blood,  the primary transport fluid in the body is about 78 percent water. Dissolved substances in blood include proteins, lipoproteins,  glucose, electrolytes, and metabolic waste products, such as carbon  dioxide and urea. These substances are either dissolved in the  

### Embedding our text chunks